In [5]:
import csv, json, os, shutil, hashlib
from pathlib import Path
from collections import Counter

# ================= STUDENT PARAMETERS =================
STUDENT_ID   = "TUPM-26-1859"
DIGITS       = [int(c) for c in STUDENT_ID if c.isdigit()]
ID_SUM       = sum(DIGITS)                    # 31
SEED_DIGIT   = DIGITS[-1]                     # 9
MAX_RECORDS  = 1000 + ID_SUM                  # 1031
TARGET_CLASS = SEED_DIGIT % 4 + 2             # 3 -> 3xx
LOW, HIGH    = TARGET_CLASS * 100, TARGET_CLASS * 100 + 99

# ================= LOCATE weblog.csv =================
FOLDER_NAME = "Lab6-Additional-Exercises"
MANUAL_PATH = r""    # optional: paste full path to weblog.csv here

def find_weblog():
    if MANUAL_PATH and Path(MANUAL_PATH).is_file():
        return Path(MANUAL_PATH)
    cwd = Path.cwd()
    for p in (cwd / "weblog.csv", cwd / FOLDER_NAME / "weblog.csv"):
        if p.is_file():
            return p
    for base in [cwd, *cwd.parents, Path.home()]:
        for hit in base.rglob(f"{FOLDER_NAME}/weblog.csv"):
            if ".venv" not in hit.parts and hit.is_file():
                return hit
    for base in [cwd, *cwd.parents, Path.home()]:
        for hit in base.rglob("weblog.csv"):
            if ".venv" not in hit.parts and "output" not in hit.parts and hit.is_file():
                return hit
    return None

SRC = find_weblog()
if SRC is None:
    raise FileNotFoundError(f"weblog.csv not found. Working dir: {Path.cwd()}\nFiles: {os.listdir()}")
SRC = SRC.resolve()
PROJECT = SRC.parent

OUT_DIR, BAK_DIR = PROJECT / "output", PROJECT / "backup"
OUT_DIR.mkdir(exist_ok=True); BAK_DIR.mkdir(exist_ok=True)
FILTERED_CSV = OUT_DIR / f"weblog_{TARGET_CLASS}xx_filtered.csv"
SUMMARY_JSON = OUT_DIR / f"weblog_{TARGET_CLASS}xx_summary.json"
BACKUP_JSON  = BAK_DIR / f"weblog_{TARGET_CLASS}xx_summary_backup.json"

# ================= READ LINE BY LINE =================
def col(header, *names):
    lk = {h.strip().lower(): h for h in header}
    for n in names:
        if n.lower() in lk:
            return lk[n.lower()]
    raise KeyError(f"Missing column {names} in {header}")

processed = valid = invalid = 0
status_counts, url_counts = Counter(), Counter()

with open(SRC, newline="", encoding="utf-8-sig") as f_in, \
     open(FILTERED_CSV, "w", newline="", encoding="utf-8") as f_out:
    reader = csv.DictReader(f_in)
    c_ip, c_t = col(reader.fieldnames, "IP"), col(reader.fieldnames, "Time")
    c_u, c_s  = col(reader.fieldnames, "URL"), col(reader.fieldnames, "Status", "Staus")
    writer = csv.writer(f_out)
    writer.writerow(["IP", "Time", "URL", "Status"])

    for row in reader:
        if processed >= MAX_RECORDS:
            break
        processed += 1
        ip, t = (row.get(c_ip) or "").strip(), (row.get(c_t) or "").strip()
        url, st = (row.get(c_u) or "").strip(), (row.get(c_s) or "").strip()
        try:
            code = int(st)
            if not (100 <= code <= 599) or not (ip and t and url):
                raise ValueError
        except (ValueError, TypeError):
            invalid += 1
            continue
        valid += 1
        if LOW <= code <= HIGH:
            writer.writerow([ip, t, url, code])
            status_counts[code] += 1
            url_counts[url] += 1




in_class = sum(status_counts.values())
top5 = url_counts.most_common(5)

# ================= JSON SUMMARY + BACKUP =================
summary = {
    "student_id": STUDENT_ID, "source_file": str(SRC),
    "records_processed": processed, "valid_records": valid, "invalid_records_skipped": invalid,
    "target_status_class": f"{TARGET_CLASS}xx", "records_in_target_class": in_class,
    "count_per_status_code": {str(k): v for k, v in sorted(status_counts.items())},
    "top_5_urls": [{"rank": i, "url": u, "count": n} for i, (u, n) in enumerate(top5, 1)],
    "filtered_csv_file": str(FILTERED_CSV),
}
with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

shutil.copy2(SUMMARY_JSON, BACKUP_JSON)
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
backup_ok = BACKUP_JSON.exists() and sha(SUMMARY_JSON) == sha(BACKUP_JSON)

# ================= VALIDATE FILTERED CSV =================
with open(FILTERED_CSV, newline="", encoding="utf-8") as f:
    rows_chk = list(csv.DictReader(f))
csv_ok = len(rows_chk) == in_class and all(LOW <= int(r["Status"]) <= HIGH for r in rows_chk)

# ================= FINAL OUTPUT =================
print("=" * 70)
print(f"ASSESSMENT DATA  |  {STUDENT_ID}  (ID_SUM={ID_SUM}, SEED_DIGIT={SEED_DIGIT})")
print("=" * 70)
print(f"Source file              : {SRC}")
print(f"Records processed        : {processed}  (valid: {valid}, invalid/skipped: {invalid})")
print(f"Target status class      : {TARGET_CLASS}xx ({LOW}-{HIGH})")
print(f"Records in target class  : {in_class}")
print("Count per status code    :")
for k, v in sorted(status_counts.items()):
    print(f"      {k}: {v}")
print("Five most requested URLs :")
for i, (u, n) in enumerate(top5, 1):
    print(f"      {i}. {u}  ->  {n}")
print(f"Filtered CSV file        : {FILTERED_CSV}  [{'VALID' if csv_ok else 'INVALID'}]")
print(f"JSON summary file        : {SUMMARY_JSON}")
print(f"Backup verification      : {'PASSED' if backup_ok else 'FAILED'} -> {BACKUP_JSON}")
print("=" * 70)


print("\nFinal Output:")
print("-" * 70)
print(f"Records processed        : {processed}")
print(f"Target status class      : {TARGET_CLASS}xx")
print(f"Records in target class  : {in_class}")
print(f"Count per status code    : {dict(sorted(status_counts.items()))}")
print("Five most requested URLs :")
for i, (u, n) in enumerate(top5, 1):
    print(f"   {i}. {u} ({n})")
print(f"Filtered CSV file        : {FILTERED_CSV.name}")
print(f"JSON summary file        : {SUMMARY_JSON.name}")
print(f"Backup verification      : {'PASSED' if backup_ok else 'FAILED'}")
print("-" * 70)
print(f"RESULT: Of the first {processed} requests, {in_class} were {TARGET_CLASS}xx responses "
      f"({in_class / processed * 100:.1f}%). Most requested URL in this class: {top5[0][0]} ({top5[0][1]} requests).")

ASSESSMENT DATA  |  TUPM-26-1859  (ID_SUM=31, SEED_DIGIT=9)
Source file              : C:\Users\jullm\Lab6_File_Handling_Pacudan\Lab6-Additional-Exercises\weblog.csv
Records processed        : 1031  (valid: 1031, invalid/skipped: 0)
Target status class      : 3xx (300-399)
Records in target class  : 166
Count per status code    :
      302: 145
      304: 21
Five most requested URLs :
      1. GET / HTTP/1.1  ->  47
      2. POST /process.php HTTP/1.1  ->  35
      3. GET /contestproblem.php?name=RUET%20OJ%20Server%20Testing%20Contest HTTP/1.1  ->  34
      4. GET /home.php HTTP/1.1  ->  9
      5. GET /logout.php HTTP/1.1  ->  7
Filtered CSV file        : C:\Users\jullm\Lab6_File_Handling_Pacudan\Lab6-Additional-Exercises\output\weblog_3xx_filtered.csv  [VALID]
JSON summary file        : C:\Users\jullm\Lab6_File_Handling_Pacudan\Lab6-Additional-Exercises\output\weblog_3xx_summary.json
Backup verification      : PASSED -> C:\Users\jullm\Lab6_File_Handling_Pacudan\Lab6-Additional-Exerci